In [1]:
import os
import pandas as pd
import glob
from datetime import datetime
import gzip
import re
import tarfile
import shutil

### Definitions

In [ ]:
# Function to filter each CSV file
def filter_csv(gz_file, prefixes, region):
	# Read the CSV file into a pandas DataFrame
	with gzip.open(gz_file, 'rt') as f:
		data = pd.read_csv(f, dtype=data_types, sep="|")
	
	# Print column names for debugging
	normalized_path = os.path.normpath(gz_file)
	print(f"Column names in {os.path.basename(normalized_path)}: {', '.join(data.columns)}")
	
	# Apply the filter to keep rows where the origin and destination columns start with specified prefixes
	filtered_data = data[
		data["origen"].str.startswith(tuple(prefixes)) &
		data["destino"].str.startswith(tuple(prefixes))
	]
	
	# Define the output file name using the gz_file path
	output_file = os.path.splitext(os.path.basename(normalized_path))[0] + "_" + region + ".csv"
	
	# Write the filtered data to a new CSV file
	filtered_data.to_csv(os.path.join(filtered_folder, output_file), index=False, encoding = "utf-8")

# Function to aggregate days
def aggregate_days(filtered_folder, output_file):
	file_list = glob.glob(os.path.join(filtered_folder, "*.csv"))
	aggregated_file = os.path.join(output_folder, output_file)
	if os.path.exists(aggregated_file):
		os.remove(aggregated_file)
	
	# Initialize a file for writing results
	write_header = True  # Control header writing for the first batch

	for file in file_list:
		file_name = os.path.basename(file)
		date_str = file_name[:8]  # Assuming the first 8 characters are the date
		file_date = datetime.strptime(date_str, "%Y%m%d")
		weekday_flag = 1 if file_date.weekday() < 5 else 0

		# Read and process each file
		data = pd.read_csv(file, dtype=data_types, sep=",")
		data['date_str'] = date_str
		data['weekday_flag'] = weekday_flag

		print(f"Trips in file: {data['viajes'].sum()}")

		# Reduce dimension by extracting province codes
		data['prov_o'] = data['origen'].str[:2]
		data['prov_d'] = data['destino'].str[:2]

		# Drop the diagonal
		print(f"Rows before filtering out diagonal: {data.shape[0]}")
		data = data[data['prov_o'] != data['prov_d']]
		print(f"Rows after filtering out diagonal: {data.shape[0]}")
		output_trips = data['viajes'].sum()
		print(f"Total trips after removing the diagonal: {output_trips}")

		# Group by criteria and aggregate
		grouping_criteria = ['prov_o', 'prov_d', 'date_str', 'weekday_flag', 'distancia', 'residencia']
		data[grouping_criteria] = data[grouping_criteria].fillna("Unknown")
		grouped_data = data.groupby(grouping_criteria).agg(
			viajes=('viajes', 'sum'),
			viajes_km=('viajes_km', 'sum')
		).reset_index()

		# Append to the output file incrementally
		grouped_data.to_csv(aggregated_file, mode='a', header=write_header, index=False, encoding = "utf-8")
		write_header = False  # Avoid writing the header again

		normalized_path = os.path.normpath(file)
		print(f"Processed {os.path.basename(normalized_path)}: {data['viajes'].sum()} trips")

	print(f"Aggregated data saved to {aggregated_file}")


### Pseudo main

##### Using old folder

In [3]:
# Define working folder. We'll work in parallel with the 3 possible subdivisions (GAU, municipality, census district)
working_folder = "D:/Documents/Proyectos/ADIF_Corredor_Córdoba_Málaga/MITMA_Jose/Inputs/Municipios/"

# Define sub-folders
gz_file_path = os.path.join(working_folder, "Zipped/")
filtered_folder = os.path.join(working_folder, "Filtered/")
output_folder = os.path.join(working_folder, "Aggregated/")
zones_shp = os.path.join(working_folder, "Zonificación_GAUs/")
# output_folder = os.path.join(working_folder, "3-Averages/")

# Define datatypes
data_types = {"origen": 'string',
              "destino": 'string', 
              "distancia": 'string',
              "actividad_origen": 'string',
              "actividad_destino": 'string',
              "residencia": 'string',
              "renta": 'string',
              "edad": 'string',
              "sexo": 'string'
}

criteria = list(data_types.keys())

# Define columns with quantities to group
# Cadiz, Ciudad Real, Córdoba, Huelva, Madrid, Málaga, 
prefixes = ["11","13","14","21","28","29","41","45"]
region = "Corredor_sur"

gz_pattern = os.path.join(gz_file_path, "*.csv.gz")
gz_file_list = glob.glob(gz_pattern)
output_file = region + "_2022-2024.csv"

# Apply the filter function to each .gz file
for gz_file in gz_file_list:
    if gz_file.endswith('.gz'):
        filter_csv(gz_file, prefixes, region)

# Aggregate the data
aggregate_days(filtered_folder, output_file)

Column names in 20240603_Viajes_municipios.csv.gz: fecha, periodo, origen, destino, distancia, actividad_origen, actividad_destino, estudio_origen_posible, estudio_destino_posible, residencia, renta, edad, sexo, viajes, viajes_km
Column names in 20240604_Viajes_municipios.csv.gz: fecha, periodo, origen, destino, distancia, actividad_origen, actividad_destino, estudio_origen_posible, estudio_destino_posible, residencia, renta, edad, sexo, viajes, viajes_km
Column names in 20240605_Viajes_municipios.csv.gz: fecha, periodo, origen, destino, distancia, actividad_origen, actividad_destino, estudio_origen_posible, estudio_destino_posible, residencia, renta, edad, sexo, viajes, viajes_km
Column names in 20240606_Viajes_municipios.csv.gz: fecha, periodo, origen, destino, distancia, actividad_origen, actividad_destino, estudio_origen_posible, estudio_destino_posible, residencia, renta, edad, sexo, viajes, viajes_km
Column names in 20240607_Viajes_municipios.csv.gz: fecha, periodo, origen, desti

### Exploratory data analysis

In [4]:
aggregated_file = os.path.join(output_folder, output_file)
print(aggregated_file)
data = pd.read_csv(aggregated_file, parse_dates=['date_str'])

D:/Documents/Proyectos/ADIF_Corredor_Córdoba_Málaga/MITMA_Jose/Inputs/Municipios/Aggregated/Corredor_sur_2022-2024.csv


In [5]:
# Evolución anual de la demanda
data['year'] = data['date_str'].dt.year
gr_data = data[['prov_o','prov_d','year','viajes']].groupby(['prov_o', 'prov_d', 'year']).sum(numeric_only=True).reset_index()
aggregated_file = os.path.join(output_folder, 'principales_relaciones.csv')
provincias = {
    11:'Cádiz',
    13:'Ciudad Real',
    14:'Córdoba',
    21:'Huelva',
    28:'Madrid',
    29:'Málaga',
    41:'Sevilla',
    45:'Toledo'
}

gr_data['pr_o'] = gr_data['prov_o'].map(provincias)
gr_data['pr_d'] = gr_data['prov_d'].map(provincias)


gr_data.to_csv(aggregated_file, index=False, encoding = "utf-8")

In [6]:
gr_data['viajes_y-1'] = gr_data.groupby(['prov_o', 'prov_d'])['viajes'].shift(1)
gr_data['YoY_growth'] = (gr_data['viajes'] - gr_data['viajes_y-1']) / gr_data['viajes_y-1'] * 100

gr_data.dropna(inplace=True)

gr_data = gr_data[gr_data['prov_o'] != 13]
gr_data = gr_data[gr_data['prov_d'] != 13]
gr_data = gr_data[gr_data['prov_o'] != 45]
gr_data = gr_data[gr_data['prov_d'] != 45]


# map province names
provincias = {
    11:'Cádiz',
    13:'Ciudad Real',
    14:'Córdoba',
    21:'Huelva',
    28:'Madrid',
    29:'Málaga',
    41:'Sevilla',
    45:'Toledo'
}

gr_data['pr_o'] = gr_data['prov_o'].map(provincias)
gr_data['pr_d'] = gr_data['prov_d'].map(provincias)

## Relaciones principales
print(f"Maximum growth: {gr_data['YoY_growth'].max()}")
print(f"Minimum growth: {gr_data['YoY_growth'].min()}")

gr_data.sort_values('YoY_growth', ascending=False, inplace=True)
display(gr_data)


# Totales por año

# Tasas de crecimiento anual

Maximum growth: 25.076918635205942
Minimum growth: -37.71291446812439


,prov_o,prov_d,year,viajes,pr_o,pr_d,viajes_y-1,YoY_growth
140,41,28,2024,80449.773,Sevilla,Madrid,64320.239000,25.076919
101,28,41,2024,77981.305,Madrid,Sevilla,64981.427000,20.005529
97,28,29,2023,73202.178,Madrid,Málaga,61766.881806,18.513637
118,29,28,2023,66472.199,Málaga,Madrid,56399.122000,17.860344
119,29,28,2024,76356.656,Málaga,Madrid,66472.199000,14.870062
86,28,11,2024,42619.036,Madrid,Cádiz,37433.810000,13.851719
91,28,14,2023,36066.479,Madrid,Córdoba,31835.979000,13.288424
113,29,14,2024,219337.760,Málaga,Córdoba,195135.228000,12.402954
11,11,28,2024,35419.868,Cádiz,Madrid,31575.748000,12.174280
56,14,29,2024,221483.700,Córdoba,Málaga,198714.353000,11.458330
